# Managed Agents API demo: step by step

> **Preview.** The Managed Agents API on Gemini Enterprise Agent Platform is a Pre-GA product.
> Do not use proprietary, sensitive, or confidential data with it. This notebook uses only
> synthetic data. Model use is billed at standard rates.

This notebook runs the same steps as the scripts in `scripts/`, in the same order:

1. Access check: create and delete one agent.
2. Demo bucket: upload `AGENTS.md`, one skill, and the synthetic sales data.
3. Agent lifecycle: create, get, list, and update the data analyst agent.
4. One task: the agent analyses the data and writes a report and a chart.
5. The report and the chart from the bucket.
6. A follow-up question in the same session.
7. A tool override for one interaction.
8. The BigQuery MCP server for one interaction.
9. A skill from Skill Registry in a second agent.
10. Cleanup.

Before you start, follow the setup in `README.md` and set `GOOGLE_CLOUD_PROJECT` in `.env`.
Run the notebook from the `managed_agents_api/` folder.

```mermaid
flowchart LR
    scripts["Demo scripts (Python SDK and REST)"] -->|"Agents API: create, get, list, update, delete"| agent["Agent configuration"]
    scripts -->|"Interactions API: tasks and follow-ups"| sandbox["Sandbox (Antigravity harness)"]
    agent --> sandbox
    bucket["Cloud Storage bucket: AGENTS.md, skill, sales.csv"] -->|"mounted at /workspace/.agent"| sandbox
    sandbox -->|"report.md and chart.png"| bucket
```

In [ ]:
from IPython.display import Image, Markdown, display
from rich.console import Console

from managed_agents_demo import (
    access_check,
    agent_config,
    cleanup,
    client,
    config,
    interact_demo,
    lifecycle_demo,
    mcp,
    skills,
    storage,
)

settings = config.load()
rest = client.RestClient(settings)  # REST: create, update, delete, operations.
sdk = client.make_genai_client(settings)  # SDK: get, list, interactions.
storage_client = storage.make_storage_client(settings.project)
console = Console(width=110)

print(f"Project: {settings.project}  Location: {settings.location}  Bucket: gs://{settings.bucket}")

## 1. Access check

The Agents API is the control plane. `CreateAgent` returns a long-running operation. The code
polls the operation until it is done, and then deletes the agent. The first agent in a project
can take three to six minutes. The next agents take about 15 seconds.

In [ ]:
assert access_check.run(rest, console) == 0

## 2. Demo bucket

The API accepts only Cloud Storage and Skill Registry sources. The code uploads three files under
the `agent/` prefix. The agent mounts this prefix at `/workspace/.agent` in its sandbox.

```mermaid
flowchart LR
    subgraph bucket["Cloud Storage: gs://BUCKET/agent/"]
        b1["AGENTS.md"]
        b2["skills/report-writer/SKILL.md"]
        b3["data/sales.csv"]
        b4["output/"]
    end
    subgraph sandbox["Sandbox: /workspace/.agent/"]
        s1["AGENTS.md"]
        s2["skills/report-writer/SKILL.md"]
        s3["data/sales.csv"]
        s4["output/report.md and the PNG chart"]
    end
    b1 -->|"mounted"| s1
    b2 -->|"mounted"| s2
    b3 -->|"mounted"| s3
    s4 -->|"written back"| b4
```

In [ ]:
result = storage.setup_bucket(
    storage_client, settings.bucket, settings.project, agent_config.bucket_files()
)
print("Bucket created" if result.created else "Bucket exists", f"gs://{result.bucket}")
for name, size in result.uploaded:
    print(f"  gs://{result.bucket}/{name}  {size} bytes")

## 3. Agent lifecycle

The agent configuration has the base agent `antigravity-preview-05-2026`, two tools, one
Cloud Storage source, and a network allowlist. The Preview API needs an allowlist when a data
source is set, and it accepts only `*`. The SDK has no `update` method, so the code sends `PATCH`.

```mermaid
sequenceDiagram
    autonumber
    participant Script as 01_lifecycle.py
    participant REST as Agents API with REST
    participant Ops as Operations API
    participant SDK as google-genai SDK
    Script->>REST: POST /agents with id, base_agent, tools, base_environment
    REST-->>Script: Operation name
    loop Every 5 s until done is true
        Script->>Ops: GET operation
        Ops-->>Script: done false or done true
    end
    Script->>SDK: client.agents.get with the agent ID
    SDK-->>Script: Configuration and effectiveIdentity
    Script->>SDK: client.agents.list with the parent
    SDK-->>Script: All agents in the project
    Script->>REST: PATCH /agents/ID with update_mask description
    REST-->>Script: Operation, done after one poll
    Note over Script,REST: 99_cleanup.py sends DELETE for each agent with the demo- prefix
```

In [ ]:
agent_config.build_agent_body(bucket=settings.bucket)

In [ ]:
assert lifecycle_demo.run(rest, sdk, console, settings) == 0

## 4. One task

The Interactions API is the data plane. The agent reads `AGENTS.md` and the `report-writer`
skill, analyses the CSV file with Python, and writes the files to `/workspace/.agent/output/`.
These files appear in the bucket. This step takes about four minutes.

In [ ]:
assert interact_demo.run_single(sdk, storage_client, console, settings) == 0

## 5. The report and the chart

The code reads the files that the agent wrote to the bucket.

In [ ]:
bucket = storage_client.bucket(settings.bucket)
for name, _ in interact_demo.list_output(storage_client, settings.bucket):
    blob = bucket.blob(name)
    if name.endswith(".md"):
        display(Markdown(blob.download_as_text()))
    elif name.endswith(".png"):
        display(Image(data=blob.download_as_bytes()))

## 6. Follow-up in the same session

The request has `previous_interaction_id` (the conversation) and `environment` (the same
sandbox, with its files). The agent answers from the first turn.

```mermaid
sequenceDiagram
    autonumber
    participant Script as 02 and 03 scripts
    participant API as Interactions API
    participant Sbx as Sandbox
    Script->>API: interactions.create with agent, input, stream True
    API-->>Script: interaction.created with the interaction ID
    API->>Sbx: Start the sandbox and mount the bucket
    loop For each agent step
        Sbx-->>API: Tool call, tool result, or text
        API-->>Script: step.start, step.delta, step.stop
    end
    API-->>Script: interaction.completed with status, usage, environment_id
    Note over Script: 02 saves both IDs in output/last_interaction.json
    Script->>API: interactions.create with previous_interaction_id and environment
    API->>Sbx: Use the same sandbox and files
    API-->>Script: Answer from the first turn
```

In [ ]:
assert interact_demo.run_session(sdk, console) == 0

## 7. Tool override for one interaction

The `tools` field of an interaction replaces the agent tools for that turn only. Here it adds
Google Search. Search results come as URL citations on the text. Then the code gets the agent
again, to show that the stored configuration did not change.

In [ ]:
assert interact_demo.run_override(sdk, console) == 0

## 8. BigQuery MCP server for one interaction

The `tools` override adds the Google-hosted BigQuery MCP server for this turn only. The request
carries a short-lived access token. The token is not stored in the agent configuration, and the
printed request masks it. MCP tool calls come as steps named `bigquery:<tool>`. Then the code runs
the same query directly with `execute_sql_readonly` and compares the results.

The account needs `roles/bigquery.jobUser` and `roles/mcp.toolUser` in the project.

In [ ]:
assert mcp.run(sdk, console, settings, token_provider=client.default_token_provider()) == 0

## 9. A skill from Skill Registry

The code publishes `assets/registry-skills/sales-chart-style/` to Skill Registry in `us-central1`.
Then it creates `demo-skill-analyst` with two sources: the demo bucket and the registry skill.
The skill requires the chart file name `chart_<metric>_by_<category>.png`. The code checks that
name in the bucket. A skill ID stays reserved after a delete.

In [ ]:
assert skills.run(rest, sdk, storage_client, console, settings) == 0

## 10. Cleanup

The code deletes every agent whose ID starts with `demo-`, and the registry skill. Set `DELETE_BUCKET = False` to keep
the bucket and the report. IAM bindings to a deleted agent identity remain active.

In [ ]:
DELETE_BUCKET = True

assert (
    cleanup.run(
        rest,
        sdk,
        storage_client,
        console,
        settings,
        yes=True,
        delete_bucket=DELETE_BUCKET,
        delete_skills=True,
        state_path=interact_demo.STATE_PATH,
    )
    == 0
)